# Phase B: Source Quality & Evidence Intelligence

**Journey:** grounded-research v2 — close completeness gap with Perplexity  
**Purpose:** Score source quality, check evidence sufficiency per sub-question, compress redundant evidence  
**Mode:** Historical planning notebook for a shipped slice  
**Plan doc:** `docs/plans/phase_b_source_quality.md`

## B.1: LLM Source Quality Scoring

**Input:** `list[SourceRecord]` with URLs + titles + snippets  
**Output:** Same list with `quality_tier` updated per-source  
**Status:** `planned`  
**Execution mode:** `stub`

**Acceptance criteria:**
- EPA/WHO/IEA → authoritative, random blogs → unknown/unreliable
- Quality tiers visible in analyst and synthesis context
- Batch scoring (one LLM call for all 50 sources)
- Cost < $0.01

In [ ]:
# Schema for batch quality scoring
from pydantic import BaseModel, Field
from typing import Literal

QualityTier = Literal["authoritative", "reliable", "unknown", "unreliable"]

class SourceQualityAssessment(BaseModel):
    """Quality assessment for one source."""
    source_id: str = Field(description="The source ID (S-xxx) being assessed.")
    quality_tier: QualityTier = Field(
        description=(
            "authoritative: government agencies, peer-reviewed journals, WHO/IEA/EPA, "
            "established think tanks. reliable: major news outlets, professional orgs. "
            "unknown: blogs, forums, unclear provenance. "
            "unreliable: misinformation, SEO farms, content mills."
        ),
    )
    reasoning: str = Field(description="One sentence explaining this tier assignment.")

class SourceQualityBatch(BaseModel):
    """Batch quality assessment for all sources."""
    assessments: list[SourceQualityAssessment]

# Stub: what we expect for the fasting evidence bundle
stub_assessments = [
    SourceQualityAssessment(source_id="S-abc", quality_tier="authoritative", reasoning="NIH/NIA government health resource"),
    SourceQualityAssessment(source_id="S-def", quality_tier="reliable", reasoning="Harvard Health Publishing — major academic institution"),
    SourceQualityAssessment(source_id="S-ghi", quality_tier="unknown", reasoning="Personal blog with no author credentials"),
]
print(f"Stub: {len(stub_assessments)} assessments")
for a in stub_assessments:
    print(f"  {a.source_id}: {a.quality_tier} — {a.reasoning}")

## B.2: Per-Sub-Question Evidence Sufficiency

**Input:** `EvidenceBundle` + `QuestionDecomposition`  
**Output:** `EvidenceBundle` with gaps added for under-covered sub-questions  
**Status:** `planned`  
**Execution mode:** `planned`

**Acceptance criteria:**
- Sub-questions with < 2 evidence items flagged as gaps
- Gap message includes the sub-question text
- Requires `EvidenceItem.sub_question_id` field (new)

**Key decision:** Evidence items are tagged with sub-question origin during query generation in collect.py. A source found via sub-question #1 may also contain evidence for #3 — but we only tag with the *search-originating* sub-question, not cross-tag. Simple, traceable, good enough.

## B.3: Conflict-Aware Compression

**Input:** `EvidenceBundle` (96+ items)  
**Output:** `EvidenceBundle` (≤ threshold items)  
**Status:** `planned`  
**Execution mode:** `planned`

**Acceptance criteria:**
- Evidence count ≤ compression_threshold (config, default 80)
- All authoritative-source evidence preserved
- Conflicting evidence preserved (both sides survive)
- At least 1 evidence item per sub-question preserved
- No broken evidence ID references downstream

**Key decision:** LLM-based compression, not embedding similarity. One call: send all evidence, get back which IDs to keep. Simple, auditable.

## Gate

Re-run fasting question with B.1 + B.2 + B.3. Fair comparison vs Perplexity deep.
- Completeness score ≥ 5 (was 4)
- No regression on decision usefulness (must stay ≥ 4)
- Cost increase < $0.02 total for all three features